# Analise de Dados - Poke API

Perguntas

- Pokémons com múltiplos tipos e força acima da média

- Abilities exclusivas de pokémons com múltiplos tipos

- Pokémons mais versáteis


## Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


## Iniciar tabelas

In [0]:
df_pokemon = spark.table("workspace.default.pokemon")
df_pokemon_estatisticas = spark.table("workspace.default.pokemon_stats")
df_pokemon_tipos = spark.table("workspace.default.pokemon_type")
df_pokemon_habilidades = spark.table("workspace.default.pokemon_ability")

# cache não habilitado no databricks serveless
# df_pokemon_habilidades = df_pokemon_habilidades.cache()
# df_pokemon_estatisticas = df_pokemon_estatisticas.cache()
# df_pokemon_tipos = df_pokemon_tipos.cache()
# df_pokemon = df_pokemon.cache()

## Pokémons com múltiplos tipos e força acima da média

In [0]:
# Cria dataframe com a força total de cada pokemons
df_pokemon_forca = (
                    df_pokemon_estatisticas
                    .groupBy("pokemon_id")
                    .agg(F.sum("base_stat").alias("total_base_stat"))
)

# Cria dataframe com a media de forças 
df_media_forca = (
            df_pokemon_forca
            .agg(F.avg("total_base_stat").alias("avg_total_base_stat")))

# cria dataframe com pokemons com mais de um tipo 
df_pokemons_multi_tipo = df_pokemon_tipos.groupBy("pokemon_id") \
                                .agg(F.count("type_name").alias("num_types"))\
                                .filter(F.col("num_types") > 1)

# 1. Comparo o total de força com a media
# 2. Filtro pokemons com força acima da media 
    # 2.1 usei windows para evitar o collect na momento de obter a media das forças 
# 3. Descarto a coluna avg_total_base_stat
# 4. Filtro pokemons com mais de um tipo
# 5. Junto com as informações de pokemon com mais de um tipo
# 6. ordeno e seleciono as colunas de interesse 
df_resultado = (
    df_pokemon_forca
    .withColumn(
        "avg_total_base_stat",
        F.avg("total_base_stat").over(Window.partitionBy())
    )
    .filter(F.col("total_base_stat") > F.col("avg_total_base_stat"))
    .drop("avg_total_base_stat")
    .join(df_pokemons_multi_tipo, on="pokemon_id", how="inner")
    .orderBy(F.col("total_base_stat").desc())
    .select("pokemon_id", "num_types", "total_base_stat")
)

print(f"{df_resultado.count()} pokemons possuem mais de um tipo e força maior que a média") 


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


510 pokemons possuem mais de um tipo e força maior que a média


## Abilities exclusivas de pokémons com múltiplos tipos

In [0]:
# número de tipos por pokemon
df_pokemon_num_tipo = (
    df_pokemon_tipos
    .groupBy("pokemon_id")
    .agg(F.countDistinct("type_name").alias("num_types"))
)

# junta com habilidades e cria coluna de filtro
df_pokemon_hab_tipo = (
    df_pokemon_habilidades
    .join(df_pokemon_num_tipo, "pokemon_id").groupBy("ability_name")
    .agg(
        F.max(F.when(F.col("num_types") > 1, 1).otherwise(0)).alias("multi_tipo"),
        F.max(F.when(F.col("num_types") == 1, 1).otherwise(0)).alias("unico_tipo")
    )
)

# pega habilidades exclusivas de multi-tipo
df_resultado = (
    df_pokemon_hab_tipo
    .filter((F.col("multi_tipo") == 1) & (F.col("unico_tipo") == 0))
    .select("ability_name")
)

display(df_resultado)

ability_name
air-lock
dancer
merciless
corrosion
punk-rock
protosynthesis
hadron-engine
hospitality
surge-surfer
aura-break


## Pokémons mais versáteis

In [0]:
df_pokemon_num_hab = df_pokemon_habilidades.groupBy("pokemon_id").agg(F.count("ability_name").alias("num_abilities"))

df_pokemon_forca_select = df_pokemon_forca.select("pokemon_id", "total_base_stat")

# Join para obter num_types, num_abilities e total_force
df_versatilidade = df_pokemon.select("pokemon_id", "name") \
    .join(df_pokemon_num_tipo, "pokemon_id") \
    .join(df_pokemon_num_hab, "pokemon_id") \
    .join(df_pokemon_forca_select, "pokemon_id")\
    .withColumn("versatility_score",(F.col("num_types") * 2) + F.col("num_abilities") + (F.col("total_base_stat") / 100)
)

# mostra os 4 pokemons com maior versatilidade
display(df_versatilidade.select("name", "versatility_score").orderBy(F.col("versatility_score").desc()).limit(5))

name,versatility_score
eternatus-eternamax,16.25
dragapult,13.0
kommo-o-totem,13.0
kommo-o,13.0
archaludon,13.0
